# 3 · Building a simple agent framework with GAME

| | Component | Class |
|---|---|---|
| **G** | Goals: what to achieve and how | `Goal` |
| **A** | Actions: what the agent can do | `Action`, `ActionRegistry` |
| **M** | Memory: what has happened | `Memory` |
| **E** | Environment: where actions run | `Environment` |

- The **GAME pieces change** from agent to agent; the **loop stays the same** → put each piece in a class, write the loop once.
- `AgentLanguage` translates GAME ↔ LLM; `Agent` runs the loop.
- Design tip: simulate the agent in a chat *before* coding it.

📄 Full notes: [`GAME_framework_notes.md`](GAME_framework_notes.md)

## Setup

In [ ]:
import os                         # files, current directory
import json                       # tool calls are exchanged as JSON
import time                       # timestamps in the environment's results
import traceback                  # full error details when an action fails
from dataclasses import dataclass, field
from typing import List, Callable, Dict, Any

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Load MiniCPM (~5 GB, downloaded on first run, then cached)
model_id = "openbmb/MiniCPM5-2B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",   # keep the precision the weights were saved in
    device_map="auto",    # use the GPU automatically when available
)

print("Model loaded successfully!")

## The prompt and the LLM call

- `Prompt` = `messages` (goals + memory) + `tools` (actions as JSON schemas).
- The course uses GPT-4o with **native function calling**; MiniCPM has none, so `generate_response` **emulates** it:
  1. tool schemas → text in the system message,
  2. generate with MiniCPM,
  3. extract the ```` ```action ```` call (tolerant: also `<action>` tags, ```` ```json ````),
  4. retry once with an actionable error.
- Same output as the GPT-4o version (`'{"tool": ..., "args": ...}'` or text) → the rest of the framework is unchanged.

In [ ]:
@dataclass
class Prompt:
    messages: List[Dict] = field(default_factory=list)   # goals + memory, as chat messages
    tools: List[Dict] = field(default_factory=list)      # actions, as JSON schemas
    metadata: dict = field(default_factory=dict)         # default_factory avoids a shared mutable default


def minicpm_chat(messages: List[Dict], max_new_tokens: int = 1024) -> str:
    """Send chat messages to MiniCPM and return its reply as text (same helper as notebooks 0-2)."""
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,   # add the "assistant's turn" marker
        enable_thinking=False,        # skip MiniCPM's hidden reasoning step
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():             # no gradients: we are not training
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    # Decode only the new tokens (drop the prompt)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)


def tools_to_instructions(tools: List[Dict]) -> str:
    """Describe the tools in plain text, since MiniCPM has no function-calling API."""
    tool_list = [
        {
            "name": t["function"]["name"],
            "description": t["function"]["description"],
            "parameters": t["function"]["parameters"],
        }
        for t in tools
    ]
    return (
        "You can use the following tools:\n"
        "```json\n" + json.dumps(tool_list, indent=2) + "\n```\n\n"
        "Every response MUST call exactly one tool. First think step by step, "
        "then write the tool call as a Markdown code block in exactly this format "
        "(a code block, not XML tags):\n"
        "```action\n"
        '{"tool": "<tool name>", "args": {<arguments>}}\n'
        "```"
    )


def extract_tool_call(text: str, tool_names: List[str]):
    """Find a tool call in the model's text. Returns {"tool": ..., "args": ...} or None.

    Tolerant: tries ```action, <action>...</action>, ```json and finally the whole text.
    """
    candidates = []
    if "```action" in text:
        candidates.append(text.split("```action", 1)[1].split("```", 1)[0])
    if "<action>" in text:
        candidates.append(text.split("<action>", 1)[1].split("</action>", 1)[0])
    if "```json" in text:
        candidates.append(text.split("```json", 1)[1].split("```", 1)[0])
    candidates.append(text)

    for candidate in candidates:
        try:
            data = json.loads(candidate.strip())
        except json.JSONDecodeError:
            continue
        if isinstance(data, dict):
            name = data.get("tool") or data.get("tool_name") or data.get("name")
            if name in tool_names:
                return {"tool": name, "args": data.get("args") or data.get("arguments") or {}}
    return None


def generate_response(prompt: Prompt, max_retries: int = 1) -> str:
    """Call the LLM. Returns a tool call as a JSON string, or plain text.

    Same output as the course's GPT-4o version:
        '{"tool": "<name>", "args": {...}}'  if the model called a tool,
        the model's text                      otherwise.
    """
    messages = [dict(m) for m in prompt.messages]          # copy: don't modify the prompt

    # No tools: just chat
    if not prompt.tools:
        return minicpm_chat(messages)

    # 1. Tools -> text instructions, appended to the system message
    instructions = tools_to_instructions(prompt.tools)
    if messages and messages[0]["role"] == "system":
        messages[0]["content"] += "\n\n" + instructions
    else:
        messages.insert(0, {"role": "system", "content": instructions})

    tool_names = [t["function"]["name"] for t in prompt.tools]

    for attempt in range(max_retries + 1):
        # 2. Generate
        text = minicpm_chat(messages)

        # 3. Extract the tool call
        call = extract_tool_call(text, tool_names)
        if call is not None:
            return json.dumps(call)

        # 4. Retry with an actionable error message
        messages += [
            {"role": "assistant", "content": text},
            {"role": "user", "content":
                "Your answer did not contain a valid tool call. Available tools: "
                + ", ".join(tool_names) + ". Reply with one tool call in this format:\n"
                '```action\n{"tool": "<tool name>", "args": {...}}\n```'},
        ]

    return text   # still no tool call: return the text (the agent language will handle it)

## G · Goals

- A goal = **what** to achieve **and how** (rules, examples of reasoning, special instructions).
- Objects instead of one "wall of text"; `priority` decides order/formatting in the prompt.

In [ ]:
@dataclass(frozen=True)          # frozen: a goal is immutable once created
class Goal:
    priority: int                # order of importance (used to sort/format goals)
    name: str                    # short label
    description: str             # what to achieve and how

## A · Actions

- `Action` = name + Python function + description + JSON-schema `parameters` + `terminal` flag.
- `ActionRegistry` = lookup *name → Action* (the LLM answers with a name).
- Different agent = different registry, same loop.

In [ ]:
class Action:
    """One tool the agent can use: its Python function + the description the LLM sees."""

    def __init__(self,
                 name: str,               # name the LLM uses to call it
                 function: Callable,      # Python function that does the work
                 description: str,        # what it does (shown to the LLM)
                 parameters: Dict,        # JSON schema of the arguments (shown to the LLM)
                 terminal: bool = False): # True -> running it ends the agent loop
        self.name = name
        self.function = function
        self.description = description
        self.terminal = terminal
        self.parameters = parameters

    def execute(self, **args) -> Any:
        """Execute the action's function"""
        return self.function(**args)


class ActionRegistry:
    """Lookup table name -> Action."""

    def __init__(self):
        self.actions = {}

    def register(self, action: Action):
        self.actions[action.name] = action

    def get_action(self, name: str) -> [Action, None]:
        return self.actions.get(name, None)          # None if the LLM invents a tool name

    def get_actions(self) -> List[Action]:
        """Get all registered actions"""
        return list(self.actions.values())

## M · Memory

- Wraps the list of messages → storage/selection strategies can change later (database, subclasses) without touching the loop.
- Items have a `type`: `user`, `assistant`, `environment`, `system`.
- ⚠️ `items[:limit]` returns the **first** items; the notes describe the **last N** (`items[-limit:]`).

In [ ]:
class Memory:
    """The agent's history: a list of dicts like {"type": "assistant", "content": "..."}."""

    def __init__(self):
        self.items = []  # basic conversation history

    def add_memory(self, memory: dict):
        """Add memory to working memory"""
        self.items.append(memory)

    def get_memories(self, limit: int = None) -> List[Dict]:
        """Get formatted conversation history for prompt"""
        # Note: items[:limit] returns the FIRST `limit` items. To get the most
        # recent ones (as the course text describes), it would be items[-limit:].
        # With limit=None (the default used here) both return everything.
        return self.items[:limit]

    def copy_without_system_memories(self):
        """Return a copy of the memory without system memories"""
        filtered_items = [m for m in self.items if m["type"] != "system"]
        memory = Memory()
        memory.items = filtered_items
        return memory

## E · Environment

- Replaces the `if/elif` chain: executes any action and returns a standard dict.
- Errors are **returned, not raised** (`tool_executed: False` + traceback) → the agent can react.

In [ ]:
class Environment:
    """Runs actions and wraps their results (or errors) in a standard dict."""

    def execute_action(self, action: Action, args: dict) -> dict:
        """Execute an action and return the result."""
        try:
            result = action.execute(**args)
            return self.format_result(result)
        except Exception as e:
            # The error goes back to the agent instead of crashing the loop
            return {
                "tool_executed": False,
                "error": str(e),
                "traceback": traceback.format_exc()
            }

    def format_result(self, result: Any) -> dict:
        """Format the result with metadata."""
        return {
            "tool_executed": True,
            "result": result,
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")
        }

## Agent language: GAME ↔ LLM

Two jobs: **construct the prompt** and **parse the response**.

| Language | Actions shown as | Parsing | Trade-off |
|---|---|---|---|
| JSON action (notebook 2 style) | text + ```` ```action ```` format | extract the block | reasoning visible, parsing fragile |
| Function calling (used here) | `tools` schemas | read the tool call | valid JSON, reasoning hidden |

- Swap the language → same agent, different prompting style.
- Change for MiniCPM: `format_memory` sends environment results as **`user`** messages.

In [ ]:
class AgentLanguage:
    """Interface: how GAME objects become a prompt, and how the reply becomes an action."""

    def __init__(self):
        pass

    def construct_prompt(self,
                         actions: List[Action],
                         environment: Environment,
                         goals: List[Goal],
                         memory: Memory) -> Prompt:
        raise NotImplementedError("Subclasses must implement this method")

    def parse_response(self, response: str) -> dict:
        raise NotImplementedError("Subclasses must implement this method")


class AgentFunctionCallingActionLanguage(AgentLanguage):
    """Language for function calling: goals -> system message, actions -> tool schemas."""

    def __init__(self):
        super().__init__()

    def format_goals(self, goals: List[Goal]) -> List:
        # All goals -> one system message: "name:\n----\ndescription\n----"
        sep = "\n-------------------\n"
        goal_instructions = "\n\n".join([f"{goal.name}:{sep}{goal.description}{sep}" for goal in goals])
        return [
            {"role": "system", "content": goal_instructions}
        ]

    def format_memory(self, memory: Memory) -> List:
        """Map memory items to chat messages."""
        # assistant decisions -> role "assistant"
        # environment results -> role "user"   (changed for MiniCPM, see above)
        # user messages       -> role "user"
        items = memory.get_memories()
        mapped_items = []
        for item in items:

            content = item.get("content", None)
            if not content:
                content = json.dumps(item, indent=4)

            if item["type"] == "assistant":
                mapped_items.append({"role": "assistant", "content": content})
            elif item["type"] == "environment":
                mapped_items.append({"role": "user", "content": content})
            else:
                mapped_items.append({"role": "user", "content": content})

        return mapped_items

    def format_actions(self, actions: List[Action]) -> [List, List]:
        """Actions -> tool schemas in the standard function-calling format."""
        tools = [
            {
                "type": "function",
                "function": {
                    "name": action.name,
                    # Include up to 1024 characters of the description
                    "description": action.description[:1024],
                    "parameters": action.parameters,
                },
            } for action in actions
        ]

        return tools

    def construct_prompt(self,
                         actions: List[Action],
                         environment: Environment,
                         goals: List[Goal],
                         memory: Memory) -> Prompt:
        # messages = goals (system) + memory (conversation); tools = actions
        prompt = []
        prompt += self.format_goals(goals)
        prompt += self.format_memory(memory)

        tools = self.format_actions(actions)

        return Prompt(messages=prompt, tools=tools)

    def adapt_prompt_after_parsing_error(self,
                                         prompt: Prompt,
                                         response: str,
                                         traceback: str,
                                         error: Any,
                                         retries_left: int) -> Prompt:
        # Hook for later: repair the prompt after a parsing error (unused for now)
        return prompt

    def parse_response(self, response: str) -> dict:
        """Parse the LLM response: a JSON tool call, or plain text -> terminate."""
        try:
            return json.loads(response)

        except Exception as e:
            # Not a tool call: treat the text as the final answer
            return {
                "tool": "terminate",
                "args": {"message": response}
            }

## The agent: the GAME loop

`agent.run(task)` → same six steps as notebook 2:

| Step | Method |
|---|---|
| 1. Construct prompt | `construct_prompt` (via `AgentLanguage`) |
| 2. Generate response | `prompt_llm_for_action` (`generate_response`) |
| 3. Parse response | `get_action` → invocation + `Action` |
| 4. Execute | `environment.execute_action` |
| 5. Update memory | `update_memory` |
| 6. Terminate? | `should_terminate` (terminal action) or `max_iterations` |

In [ ]:
class Agent:
    def __init__(self,
                 goals: List[Goal],
                 agent_language: AgentLanguage,
                 action_registry: ActionRegistry,
                 generate_response: Callable[[Prompt], str],
                 environment: Environment):
        """
        Initialize an agent with its core GAME components
        """
        self.goals = goals                          # G
        self.generate_response = generate_response  # the LLM
        self.agent_language = agent_language        # GAME <-> prompt translator
        self.actions = action_registry              # A
        self.environment = environment              # E   (M is passed to run())

    def construct_prompt(self, goals: List[Goal], memory: Memory, actions: ActionRegistry) -> Prompt:
        """Build prompt with memory context"""
        return self.agent_language.construct_prompt(
            actions=actions.get_actions(),
            environment=self.environment,
            goals=goals,
            memory=memory
        )

    def get_action(self, response):
        # Reply -> {"tool": ..., "args": ...} -> the registered Action object
        invocation = self.agent_language.parse_response(response)
        action = self.actions.get_action(invocation["tool"])
        return action, invocation

    def should_terminate(self, response: str) -> bool:
        action_def, _ = self.get_action(response)
        return action_def.terminal

    def set_current_task(self, memory: Memory, task: str):
        memory.add_memory({"type": "user", "content": task})

    def update_memory(self, memory: Memory, response: str, result: dict):
        """
        Update memory with the agent's decision and the environment's response.
        """
        new_memories = [
            {"type": "assistant", "content": response},
            {"type": "environment", "content": json.dumps(result)}
        ]
        for m in new_memories:
            memory.add_memory(m)

    def prompt_llm_for_action(self, full_prompt: Prompt) -> str:
        response = self.generate_response(full_prompt)
        return response

    def run(self, user_input: str, memory=None, max_iterations: int = 50) -> Memory:
        """
        Execute the GAME loop for this agent with a maximum iteration limit.
        """
        memory = memory or Memory()
        self.set_current_task(memory, user_input)

        for _ in range(max_iterations):
            # 1. Construct a prompt that includes the Goals, Actions, and the current Memory
            prompt = self.construct_prompt(self.goals, memory, self.actions)

            # 2. Generate a response from the agent
            print("Agent thinking...")
            response = self.prompt_llm_for_action(prompt)
            print(f"Agent Decision: {response}")

            # 3. Determine which action the agent wants to execute
            action, invocation = self.get_action(response)

            # 4. Execute the action in the environment
            result = self.environment.execute_action(action, invocation["args"])
            print(f"Action Result: {result}")

            # 5. Update the agent's memory with information about what happened
            self.update_memory(memory, response, result)

            # 6. Check if the agent has decided to terminate
            if self.should_terminate(response):
                break

        return memory

### Flow of one iteration

```
Memory + Goals + ActionRegistry ─► AgentLanguage ─► prompt ─► LLM ─► response
     ▲                                                                  │
     └─ result ◄─ Environment ◄─ Action (by name) ◄─ AgentLanguage parse ┘
```

- **action** = what the agent *can* do; **invocation** = the arguments it *chose*.
- Actions are *defined* in the registry, *executed* in the environment.

### Specialized agents

Same loop, different pieces, e.g. a research agent (web search tools, web environment) or a coding agent (write/test tools, dev environment). With tool **tags** (see notes), a registry can be built from a category: `ActionRegistry(tags=["file_operations"])`.

## Example: a README agent

- G: read every file, then terminate with the README · A: list, read, terminate · E: standard environment.
- ⚠️ `list_project_files` only lists **`.py`** files: add some to the folder first.

In [ ]:
# G: goals
goals = [
    Goal(priority=1, name="Gather Information", description="Read each file in the project"),
    Goal(priority=1, name="Terminate", description="Call the terminate call when you have read all the files "
                                                   "and provide the content of the README in the terminate message")
]

# Language: how GAME becomes a prompt
agent_language = AgentFunctionCallingActionLanguage()

# A: the Python functions behind the actions
def read_project_file(name: str) -> str:
    with open(name, "r") as f:
        return f.read()

def list_project_files() -> List[str]:
    return sorted([file for file in os.listdir(".") if file.endswith(".py")])   # only .py files

# A: register the actions (name, function, description, JSON schema, terminal?)
action_registry = ActionRegistry()
action_registry.register(Action(
    name="list_project_files",
    function=list_project_files,
    description="Lists all files in the project.",
    parameters={},
    terminal=False
))
action_registry.register(Action(
    name="read_project_file",
    function=read_project_file,
    description="Reads a file from the project.",
    parameters={
        "type": "object",
        "properties": {
            "name": {"type": "string"}
        },
        "required": ["name"]
    },
    terminal=False
))
action_registry.register(Action(
    name="terminate",
    function=lambda message: f"{message}\nTerminating...",
    description="Terminates the session and prints the message to the user.",
    parameters={
        "type": "object",
        "properties": {
            "message": {"type": "string"}
        },
        "required": []
    },
    terminal=True                       # running it ends the loop
))

# E: the environment
environment = Environment()

# The agent = GAME pieces + language + LLM
agent = Agent(goals, agent_language, action_registry, generate_response, environment)

# Run it (M: run() creates a fresh Memory)
user_input = "Write a README for this project."
final_memory = agent.run(user_input)

# Print the final memory
print(final_memory.get_memories())

### Reference run (course version, GPT-4o, no `.py` files)

```
Agent thinking...
Agent Decision: {"tool": "list_project_files", "args": {}}
Action Result: {'tool_executed': True, 'result': [], 'timestamp': '2025-03-10T20:50:18+0000'}
Agent thinking...
Agent Decision: {"tool": "terminate", "args": {"message": "It seems there are no files in the project, as the list of files returned was empty. Consequently, I cannot generate a README file content for this project since there's nothing to describe or explain. If you have more details about the project or specific files you'd like to mention, please provide them. Otherwise, the current project state does not require a README."}}
Action Result: {'tool_executed': True, 'result': "It seems there are no files in the project, as the list of files returned was empty. Consequently, I cannot generate a README file content for this project since there's nothing to describe or explain. If you have more details about the project or specific files you'd like to mention, please provide them. Otherwise, the current project state does not require a README.\nTerminating...", 'timestamp': '2025-03-10T20:50:19+0000'}
[{'type': 'user', 'content': 'Write a README for this project.'}, {'type': 'assistant', 'content': '{"tool": "list_project_files", "args": {}}'}, {'type': 'environment', 'content': '{"tool_executed": true, "result": [], "timestamp": "2025-03-10T20:50:18+0000"}'}, {'type': 'assistant', 'content': '{"tool": "terminate", "args": {"message": "It seems there are no files in the project, as the list of files returned was empty. Consequently, I cannot generate a README file content for this project since there\'s nothing to describe or explain. If you have more details about the project or specific files you\'d like to mention, please provide them. Otherwise, the current project state does not require a README."}}'}, {'type': 'environment', 'content': '{"tool_executed": true, "result": "It seems there are no files in the project, as the list of files returned was empty. Consequently, I cannot generate a README file content for this project since there\'s nothing to describe or explain. If you have more details about the project or specific files you\'d like to mention, please provide them. Otherwise, the current project state does not require a README.\\nTerminating...", "timestamp": "2025-03-10T20:50:19+0000"}'}]
```

## Key benefits

1. **Organization** – one class per component.
2. **Reusability** – swap pieces (here: GPT-4o → MiniCPM) without touching the loop.
3. **Extensibility** – a new tool is one `register(...)`.
4. **Standard interface** – every agent: `agent.run(task)` → `Memory`.
5. **Memory management** – handled inside `run()`.